# How much Data is given


In [1]:
from pathlib import Path
import json


def collect_json_files(directories):
    """Collect JSONL and JSON files from all relevant source folders."""
    collected = []
    seen = set()

    for directory in directories:
        if not directory.exists():
            print(f"Skipping missing directory: {directory}")
            continue

        for pattern in ("*.jsonl", "*.json"):
            for file_path in sorted(directory.glob(pattern)):
                resolved = file_path.resolve()
                if resolved in seen:
                    continue
                seen.add(resolved)
                collected.append(file_path)

    return collected


data_dirs = [
    Path("FT-data-20260612-original"),
    Path("(Copy) FT-data-20260522"),
]
files = collect_json_files(data_dirs)

if not files:
    print("No JSON/JSONL files found in the configured data directories.")
else:
    total_lines = 0
    total_nonempty = 0
    total_invalid = 0

    print(f"Found {len(files)} file(s):\n")

    for file_path in files:
        lines = nonempty = invalid = 0

        with file_path.open("r", encoding="utf-8-sig", errors="replace") as file:
            for line in file:
                lines += 1
                if line.strip():
                    nonempty += 1
                    try:
                        json.loads(line)
                    except json.JSONDecodeError:
                        invalid += 1

        total_lines += lines
        total_nonempty += nonempty
        total_invalid += invalid

        print(
            f"{file_path.name}: "
            f"{lines:,} lines, "
            f"{nonempty:,} non-empty records, "
            f"{invalid:,} invalid JSON lines, "
            f"{file_path.stat().st_size:,} bytes"
        )

    print("\nTotals:")
    print(f"Files: {len(files):,}")
    print(f"Lines: {total_lines:,}")
    print(f"Non-empty records: {total_nonempty:,}")
    print(f"Invalid JSON lines: {total_invalid:,}")

Found 18 file(s):

Ablauf1 100626.jsonl: 9,317 lines, 9,317 non-empty records, 0 invalid JSON lines, 39,246,240 bytes
Ablauf2 100626.jsonl: 8,547 lines, 8,547 non-empty records, 0 invalid JSON lines, 36,322,625 bytes
Ablauf2.1 08072026.jsonl: 10,012 lines, 10,012 non-empty records, 0 invalid JSON lines, 48,085,982 bytes
Ablauf3 100626.jsonl: 4,227 lines, 4,227 non-empty records, 0 invalid JSON lines, 17,183,723 bytes
Ablauf4 100626.jsonl: 8,014 lines, 8,014 non-empty records, 0 invalid JSON lines, 35,971,590 bytes
Ablauf1 100626.json: 9,317 lines, 9,317 non-empty records, 0 invalid JSON lines, 39,869,930 bytes
Ablauf2 100626.json: 8,547 lines, 8,547 non-empty records, 0 invalid JSON lines, 36,919,263 bytes
Ablauf2.1 08072026.json: 10,012 lines, 10,012 non-empty records, 0 invalid JSON lines, 48,725,929 bytes
Ablauf3 100626.json: 4,227 lines, 4,227 non-empty records, 0 invalid JSON lines, 17,451,990 bytes
Ablauf4 100626.json: 8,014 lines, 8,014 non-empty records, 0 invalid JSON lines, 3

# How long does each log take?

In [ ]:
from datetime import datetime, timezone

timestamp_keys = {
    "timestamp", "time", "@timestamp",
    "created_at", "updated_at", "datetime", "date"
}

def parse_timestamp(value):
    if isinstance(value, (int, float)):
        # Treat large numeric values as milliseconds since epoch.
        seconds = value / 1000 if value > 10_000_000_000 else value
        return datetime.fromtimestamp(seconds, tz=timezone.utc)

    if isinstance(value, str):
        try:
            parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
            return parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)
        except ValueError:
            return None

    return None

def find_timestamp(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key.lower() in timestamp_keys:
                timestamp = parse_timestamp(value)
                if timestamp:
                    return timestamp
        for value in obj.values():
            timestamp = find_timestamp(value)
            if timestamp:
                return timestamp

    elif isinstance(obj, list):
        for value in obj:
            timestamp = find_timestamp(value)
            if timestamp:
                return timestamp

    return None

for file_path in files:
    if file_path.suffix.lower() != ".jsonl":
        continue

    first_timestamp = None
    last_timestamp = None
    timestamp_count = 0

    with file_path.open("r", encoding="utf-8-sig", errors="replace") as file:
        for line in file:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            timestamp = find_timestamp(record)
            if timestamp:
                if first_timestamp is None:
                    first_timestamp = timestamp
                last_timestamp = timestamp
                timestamp_count += 1

    if first_timestamp and last_timestamp:
        duration = last_timestamp - first_timestamp
        print(
            f"{file_path.name}: "
            f"{first_timestamp.isoformat()} -> {last_timestamp.isoformat()} "
            f"({duration}, {timestamp_count:,} timestamps)"
        )
    else:
        print(f"{file_path.name}: no timestamps found")

NameError: name 'find_timestamp' is not defined

# Zeitbasierte Ereignis- und Daueranalyse

Die folgenden Zellen extrahieren fachliche Zeitstempel und Zustandswechsel aus den JSONL-Aufnahmen. Der äußere JSONL-Zeitstempel wird nur als technische Fallback-Zeit verwendet. Ein fertiges Event Log wird noch nicht geschrieben; `event_records` ist bereits für die spätere Log-Erzeugung vorbereitet.

In [ ]:
from collections import Counter, defaultdict


def parse_event_time(value):
    if isinstance(value, str):
        try:
            parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
            return parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)
        except ValueError:
            return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        seconds = value / 1000 if value > 10_000_000_000 else value
        try:
            return datetime.fromtimestamp(seconds, tz=timezone.utc)
        except (OverflowError, OSError, ValueError):
            return None
    return None


def payload_time(payload):
    if not isinstance(payload, dict):
        return None, None
    for key in ("timestamp", "ts"):
        parsed = parse_event_time(payload.get(key))
        if parsed is not None:
            return parsed, key
    return None, None


def iso(value):
    return value.isoformat() if value is not None else None


def load_records_from_file(file_path):
    """Support both JSON Lines files and regular JSON files (single object or array)."""
    text = file_path.read_text(encoding="utf-8-sig", errors="replace")
    if not text.strip():
        return []

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        rows = []
        for line_number, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                invalid_records.append((file_path.name, line_number))
                continue
            if isinstance(record, dict):
                rows.append((line_number, record))
        return rows

    if isinstance(parsed, list):
        items = parsed
    elif isinstance(parsed, dict):
        items = [parsed]
    else:
        return []

    return [(1, item) for item in items if isinstance(item, dict)]


records = []
invalid_records = []
for file_path in files:
    for line_number, record in load_records_from_file(file_path):
        payload = record.get("payload")
        source_time, source_field = payload_time(payload)
        outer_time = parse_event_time(record.get("timestamp"))
        records.append({
            "file": file_path.name,
            "line": line_number,
            "topic": str(record.get("topic", "")),
            "payload": payload,
            "outer_timestamp": outer_time,
            "source_timestamp": source_time,
            "source_timestamp_field": source_field,
        })

records.sort(key=lambda record: (
    record["source_timestamp"] is None,
    record["source_timestamp"] or record["outer_timestamp"]
))

topic_timestamp_counts = Counter(
    record["topic"] for record in records if record["source_timestamp"] is not None
)
print(f"Records geladen: {len(records):,}")
print(f"Ungültige JSONL-/JSON-Einträge: {len(invalid_records):,}")
print("Topics mit fachlichen Zeitstempeln:")
for topic, count in topic_timestamp_counts.most_common(20):
    print(f"{count:>6,}  {topic}")

Records geladen: 122,824
Ungültige JSONL-/JSON-Einträge: 0
Topics mit fachlichen Zeitstempeln:
45,657  /j1/txt/1/i/cam
21,469  module/v1/ff/SVR4H92774/instantAction
 4,947  fts/v1/ff/ylK4/state
 4,467  fts/v1/ff/1mO4/state
 2,199  /j1/txt/1/f/i/order
 1,669  module/v1/ff/SVR4H73901/connection
 1,669  module/v1/ff/SVR4H92774/connection
 1,667  module/v1/ff/SVR4H88474/connection
 1,667  module/v1/ff/SVR3QL1089/connection
 1,667  module/v1/ff/SVR4H77011/connection
 1,433  module/v1/ff/NodeRed/SVR4H92774/state
 1,081  ccu/state/stock
 1,081  /j1/txt/1/f/i/stock
   803  module/v1/ff/SVR4H92774/state
   695  /j1/txt/1/i/ldr
   682  /j1/txt/1/i/bme680
   593  module/v1/ff/SVR4H73901/state
   457  module/v1/ff/NodeRed/SVR4H92774/order
   373  fts/v1/ff/ylK4/instantAction
   356  module/v1/ff/SVR3QL1089/state


In [ ]:
# Datencontainer für die spätere Event-Log-Erzeugung
# Jede Liste enthält bereits normalisierte Records mit datetime-Objekten.
event_records = []
module_action_records = []
fts_state_records = []
fts_order_records = []
error_records = []
order_request_records = []
order_response_records = []
completed_order_records = []


def record_time(record):
    """Nutze die fachliche Payload-Zeit und nur ersatzweise die Capture-Zeit."""
    return record["source_timestamp"] or record["outer_timestamp"]


def add_event(*, record, timestamp, activity, case_id=None, resource=None,
              lifecycle=None, order_id=None, order_update_id=None,
              action_id=None, workpiece_id=None, event_kind=None, details=None,
              destination_node=None, destination_sequence=None,
              current_node=None, current_sequence=None, location=None):
    """Füge einen einheitlichen, später exportierbaren Event-Datensatz hinzu."""
    event_records.append({
        "case_id": case_id,
        "activity": activity,
        "timestamp": timestamp,
        "resource": resource,
        "lifecycle": lifecycle,
        "order_id": order_id,
        "order_update_id": order_update_id,
        "action_id": action_id,
        "workpiece_id": workpiece_id,
        "event_kind": event_kind,
        "source_topic": record["topic"],
        "source_file": record["file"],
        "source_line": record["line"],
        "destination_node": destination_node,
        "destination_sequence": destination_sequence,
        "current_node": current_node,
        "current_sequence": current_sequence,
        "location": location,
        "details": details or {},
    })

print("Event-Container initialisiert.")
print("Zielspalten:", "case_id, activity, timestamp, resource, lifecycle, order_id, action_id")
print("Zusatzattribute:", "destination_node, destination_sequence, current_node, current_sequence, location")


Event-Container initialisiert.
Zielspalten: case_id, activity, timestamp, resource, lifecycle, order_id, action_id
Zusatzattribute: destination_node, destination_sequence, current_node, current_sequence, location


In [ ]:
# CCU-Aufträge: Requests, Responses und Status-Snapshots sammeln.
for record in records:
    topic = record["topic"]
    payload = record["payload"]
    timestamp = record_time(record)
    if timestamp is None or payload is None:
        continue

    if topic == "ccu/order/request" and isinstance(payload, dict):
        order_request_records.append({"record": record, "timestamp": timestamp, **payload})
        add_event(
            record=record,
            timestamp=timestamp,
            activity="ORDER_REQUEST",
            case_id=payload.get("workpieceId"),
            workpiece_id=payload.get("workpieceId"),
            event_kind="ccu_order_request",
            details={"order_type": payload.get("orderType"), "type": payload.get("type")},
        )

    elif topic == "ccu/order/response" and isinstance(payload, dict):
        order_response_records.append({"record": record, "timestamp": timestamp, **payload})
        add_event(
            record=record,
            timestamp=timestamp,
            activity="ORDER_RESPONSE",
            case_id=payload.get("orderId"),
            order_id=payload.get("orderId"),
            workpiece_id=payload.get("workpieceId"),
            event_kind="ccu_order_response",
            details={"state": payload.get("state")},
        )

    elif topic in {"ccu/order/active", "ccu/order/completed"} and isinstance(payload, list):
        for order in payload:
            if not isinstance(order, dict):
                continue
            order_id = order.get("orderId")
            if topic.endswith("completed"):
                completed_order_records.append({"record": record, "timestamp": timestamp, **order})
            for field, activity in (("receivedAt", "ORDER_RECEIVED"), ("startedAt", "ORDER_STARTED"), ("stoppedAt", "ORDER_STOPPED")):
                event_time = parse_event_time(order.get(field))
                if event_time is not None:
                    add_event(
                        record=record,
                        timestamp=event_time,
                        activity=activity,
                        case_id=order_id,
                        order_id=order_id,
                        workpiece_id=order.get("workpieceId"),
                        lifecycle="complete" if field == "stoppedAt" else "start" if field == "startedAt" else None,
                        event_kind="ccu_order_snapshot",
                        details={"order_state": order.get("state"), "snapshot_topic": topic},
                    )

print("CCU-Extraktion abgeschlossen:")
print(f"  Requests: {len(order_request_records):,}")
print(f"  Responses: {len(order_response_records):,}")
print(f"  Completed-Snapshots: {len(completed_order_records):,}")

CCU-Extraktion abgeschlossen:
  Requests: 258
  Responses: 256
  Completed-Snapshots: 1,287


In [ ]:
# CCU-Payloads enthalten Zeitfelder als Strings; für Differenzen werden sie hier zu datetime konvertiert.
for row in order_request_records + order_response_records + completed_order_records:
    row["timestamp"] = parse_event_time(row.get("timestamp")) or record_time(row["record"])

print("CCU-Zeitfelder normalisiert:", len(order_request_records), "Requests,", len(order_response_records), "Responses")

CCU-Zeitfelder normalisiert: 258 Requests, 256 Responses


## 2. Modulaktionen extrahieren

Modulaktionen werden über `action.id`, `command` und `state` gruppiert. Die Zustände `WAITING`, `INITIALIZING`, `RUNNING`, `FINISHED` und `FAILED` liefern die fachlichen Start- und Endpunkte.

In [ ]:
# Modul-State-Nachrichten in einzelne Action-State-Events umwandeln.
for record in records:
    topic = record["topic"]
    payload = record["payload"]
    timestamp = record_time(record)
    if not (topic.startswith("module/") and topic.endswith("/state") and isinstance(payload, dict) and timestamp):
        continue

    module_id = payload.get("serialNumber") or topic.split("/")[-2]
    actions = payload.get("actionStates") or []
    if not actions and isinstance(payload.get("actionState"), dict):
        actions = [payload["actionState"]]

    for action in actions:
        if not isinstance(action, dict):
            continue

        command = str(action.get("command") or "UNKNOWN")
        if "factsheet" in command.lower():
            continue

        action_time = parse_event_time(action.get("timestamp")) or timestamp
        state = str(action.get("state", "UNKNOWN")).upper().replace("WAITIING", "WAITING")
        module_action_records.append({
            "timestamp": action_time,
            "module": module_id,
            "order_id": payload.get("orderId"),
            "order_update_id": payload.get("orderUpdateId"),
            "action_id": action.get("id"),
            "command": action.get("command"),
            "state": state,
            "result": action.get("result"),
            "record": record,
        })
        add_event(
            record=record,
            timestamp=action_time,
            activity=f"MODULE_{action.get('command', 'UNKNOWN')}_{state}",
            case_id=payload.get("orderId") or action.get("id"),
            resource=module_id,
            lifecycle=state,
            order_id=payload.get("orderId"),
            order_update_id=payload.get("orderUpdateId"),
            action_id=action.get("id"),
            event_kind="module_action_state",
            details={"command": action.get("command"), "result": action.get("result")},
        )

print(f"Modul-Action-State-Events: {len(module_action_records):,}")
print("Beispielzustände:", sorted({row["state"] for row in module_action_records}))


Modul-Action-State-Events: 3,618
Beispielzustände: ['FAILED', 'FINISHED', 'RUNNING', 'WAITING']


## 3. FTS-Navigation, Übergaben und Fehler extrahieren

Für den FTS werden zwei Zustandswechsel benötigt: `driving` für die Fahrt und `waitingForLoadHandling` für die Werkstückübergabe. Fehler werden zusätzlich als eigene Events gespeichert.

In [ ]:
# FTS-Aufträge, FTS-Zustände und Fehler getrennt erfassen.
for record in records:
    topic = record["topic"]
    payload = record["payload"]
    timestamp = record_time(record)
    if timestamp is None or not topic.startswith("fts/") or not isinstance(payload, dict):
        continue

    fts_id = payload.get("serialNumber") or topic.split("/")[-2]

    if topic.endswith("/order"):
        nodes = payload.get("nodes") or []
        node_ids = [node.get("id") for node in nodes if isinstance(node, dict) and node.get("id")]
        target_node = node_ids[-1] if node_ids else None
        target_sequence = len(node_ids) if node_ids else None
        target_node_data = nodes[-1] if nodes and isinstance(nodes[-1], dict) else {}
        fts_order_records.append({
            "timestamp": timestamp,
            "fts": fts_id,
            "order_id": payload.get("orderId"),
            "order_update_id": payload.get("orderUpdateId"),
            "target": target_node,
            "target_sequence": target_sequence,
            "node_ids": node_ids,
            "record": record,
        })
        add_event(
            record=record,
            timestamp=timestamp,
            activity="FTS_ORDER_SENT",
            case_id=payload.get("orderId"),
            resource=fts_id,
            order_id=payload.get("orderId"),
            order_update_id=payload.get("orderUpdateId"),
            event_kind="fts_order",
            destination_node=target_node,
            destination_sequence=target_sequence,
            location=target_node,
            details={
                "destination_node": target_node,
                "destination_sequence": target_sequence,
                "route_node_ids": node_ids,
                "destination": target_node_data,
                "nodes": nodes,
            },
        )

    elif topic.endswith("/state"):
        current_node = payload.get("lastNodeId")
        current_sequence = payload.get("lastNodeSequenceId")
        state_row = {
            "timestamp": timestamp,
            "fts": fts_id,
            "order_id": payload.get("orderId"),
            "order_update_id": payload.get("orderUpdateId"),
            "driving": payload.get("driving"),
            "waiting_for_load_handling": payload.get("waitingForLoadHandling"),
            "last_node_id": current_node,
            "last_node_sequence_id": current_sequence,
            "load": payload.get("load"),
            "record": record,
        }
        fts_state_records.append(state_row)

    elif topic.endswith("/instantAction"):
        for action in payload.get("actions") or []:
            if isinstance(action, dict):
                action_type = str(action.get("actionType") or "UNKNOWN")
                if "factsheet" in action_type.lower():
                    continue
                add_event(
                    record=record,
                    timestamp=timestamp,
                    activity=f"FTS_{action_type}",
                    case_id=action.get("actionId"),
                    resource=fts_id,
                    action_id=action.get("actionId"),
                    event_kind="fts_instant_action",
                    details=action,
                )

    if topic.endswith("/state"):
        for error in payload.get("errors") or []:
            if not isinstance(error, dict):
                continue
            error_time = parse_event_time(error.get("timestamp")) or timestamp
            error_row = {
                "timestamp": error_time,
                "device": fts_id,
                "device_type": "fts",
                "error_type": error.get("errorType"),
                "error_level": error.get("errorLevel"),
                "references": error.get("errorReferences", []),
                "order_id": payload.get("orderId"),
                "record": record,
            }
            error_records.append(error_row)
            add_event(
                record=record,
                timestamp=error_time,
                activity=f"ERROR_{error.get('errorType', 'UNKNOWN')}",
                case_id=payload.get("orderId"),
                resource=fts_id,
                order_id=payload.get("orderId"),
                event_kind="fts_error",
                current_node=payload.get("lastNodeId"),
                current_sequence=payload.get("lastNodeSequenceId"),
                location=payload.get("lastNodeId"),
                details={"error_level": error.get("errorLevel"), "references": error.get("errorReferences", [])},
            )

print("FTS-/Fehler-Extraktion abgeschlossen:")
print(f"  FTS-Orders: {len(fts_order_records):,}")
print(f"  FTS-States: {len(fts_state_records):,}")
print(f"  Fehler: {len(error_records):,}")


FTS-/Fehler-Extraktion abgeschlossen:
  FTS-Orders: 667
  FTS-States: 9,414
  Fehler: 91


## 1. CCU-Aufträge extrahieren

Die CCU-Daten bilden den übergeordneten Produktionsfall. Hier werden Requests, Responses sowie die Status-Snapshots `active` und `completed` gesammelt. Die Felder `orderId`, `workpieceId`, `receivedAt`, `startedAt` und `stoppedAt` werden später für die Auftragsdauer verwendet.

In [ ]:
def seconds_between(start, end):
    """Berechne eine Dauer in Sekunden; fehlende Endpunkte bleiben sichtbar."""
    if start is None or end is None:
        return None
    return round((end - start).total_seconds(), 3)


def compact_row(row, fields):
    """Zeige nur die für die jeweilige Analyse relevanten Spalten."""
    return {field: row.get(field) for field in fields}


def summarize_production_steps(order):
    """Ordne die Produktionsschritte nach ihrem beobachteten Abschlusszustand ein."""
    performed_states = {"FINISHED", "COMPLETED", "PASSED", "SUCCESS", "SUCCEEDED"}
    failed_states = {"FAILED", "ERROR", "CANCELLED", "CANCELED", "ABORTED"}
    not_started_states = {"ENQUEUED", "WAITING", "PENDING", "SKIPPED", "NOT_STARTED"}

    performed = []
    failed = []
    not_performed = []
    unknown = []

    for index, step in enumerate(order.get("productionSteps") or [], start=1):
        if not isinstance(step, dict):
            continue
        state = str(step.get("state") or "UNKNOWN").upper()
        description = {
            "position": index,
            "step_id": step.get("id"),
            "type": step.get("type"),
            "command": step.get("command"),
            "module_type": step.get("moduleType"),
            "serial_number": step.get("serialNumber"),
            "state": state,
            "started_at": step.get("startedAt"),
            "stopped_at": step.get("stoppedAt"),
        }

        if state in performed_states:
            performed.append(description)
        elif state in failed_states:
            failed.append(description)
        elif state in not_started_states:
            not_performed.append(description)
        else:
            unknown.append(description)

    # Ein Auftrag mit ERROR kann Schritte enthalten, die gar nicht gestartet wurden.
    # Diese bleiben getrennt von tatsächlich fehlgeschlagenen Schritten.
    return {
        "performed": performed,
        "failed": failed,
        "not_performed": not_performed,
        "unknown": unknown,
    }


# Die Payload-Zeitstempel der CCU-Requests und Responses werden vereinheitlicht.
for row in order_request_records + order_response_records + completed_order_records:
    row["timestamp"] = parse_event_time(row.get("timestamp")) or record_time(row["record"])

# Pro Auftrag wird der letzte completed-Snapshot verwendet.
latest_completed = {}
for row in completed_order_records:
    order_id = row.get("orderId")
    if not order_id:
        continue
    old = latest_completed.get(order_id)
    if old is None or (row.get("stoppedAt") or "") > (old.get("stoppedAt") or ""):
        latest_completed[order_id] = row

order_duration_rows = []
for order_id, order in latest_completed.items():
    received_at = parse_event_time(order.get("receivedAt"))
    started_at = parse_event_time(order.get("startedAt"))
    stopped_at = parse_event_time(order.get("stoppedAt"))
    step_summary = summarize_production_steps(order)
    order_duration_rows.append({
        "order_id": order_id,
        "workpiece_id": order.get("workpieceId"),
        "type": order.get("type"),
        "order_type": order.get("orderType"),
        "state": order.get("state"),
        "received_at": received_at,
        "started_at": started_at,
        "stopped_at": stopped_at,
        "queue_duration_s": seconds_between(received_at, started_at),
        "processing_duration_s": seconds_between(started_at, stopped_at),
        "total_duration_s": seconds_between(received_at, stopped_at),
        "step_count": len(order.get("productionSteps") or []),
        "performed_steps": step_summary["performed"],
        "failed_steps": step_summary["failed"],
        "not_performed_steps": step_summary["not_performed"],
        "unknown_steps": step_summary["unknown"],
        "performed_step_names": [step.get("command") or step.get("type") for step in step_summary["performed"]],
        "failed_step_names": [step.get("command") or step.get("type") for step in step_summary["failed"]],
        "not_performed_step_names": [step.get("command") or step.get("type") for step in step_summary["not_performed"]],
        "has_step_error": bool(step_summary["failed"]),
        "has_unperformed_steps": bool(step_summary["not_performed"]),
    })

order_duration_rows.sort(key=lambda row: row["started_at"] or datetime.max.replace(tzinfo=timezone.utc))

print("CCU-Auftragsanalyse")
print(f"Vollständig beobachtete Aufträge: {len(order_duration_rows):,}")
print("Dauer = stoppedAt - startedAt; Wartezeit = startedAt - receivedAt")
print("Schrittklassifikation: FINISHED/PASSED = durchgeführt; FAILED/ERROR = fehlgeschlagen; ENQUEUED/WAITING/SKIPPED = nicht durchgeführt")

for row in order_duration_rows:
    print("\nAuftrag:", row["order_id"])
    print("  Typ/Status:", row["type"], "/", row["state"])
    print("  Dauer (s):", row["processing_duration_s"], "| Gesamt (s):", row["total_duration_s"])
    print("  Durchgeführt:", row["performed_step_names"] or "keine")
    print("  Fehlgeschlagen:", row["failed_step_names"] or "keine")
    print("  Nicht durchgeführt:", row["not_performed_step_names"] or "keine")
    if row["unknown_steps"]:
        print("  Unklarer Schrittstatus:", [step.get("command") or step.get("type") for step in row["unknown_steps"]])

# Requests besitzen in den Aufnahmen keine orderId. Daher ist dies nur ein markierter Näherungsmatch.
request_response_rows = []
for request in order_request_records:
    candidates = [
        response for response in order_response_records
        if response["record"]["file"] == request["record"]["file"]
        and response["timestamp"] >= request["timestamp"]
        and response.get("type") == request.get("type")
        and response.get("orderType") == request.get("orderType")
    ]
    if candidates:
        response = min(candidates, key=lambda item: item["timestamp"])
        request_response_rows.append({
            "file": request["record"]["file"],
            "request_timestamp": request["timestamp"],
            "response_timestamp": response["timestamp"],
            "request_to_response_s": seconds_between(request["timestamp"], response["timestamp"]),
            "type": request.get("type"),
            "order_type": request.get("orderType"),
            "order_id": response.get("orderId"),
            "match_quality": "chronological_type_match_without_request_order_id",
        })

print(f"\nRequest-Response-Näherungsmatches: {len(request_response_rows):,} von {len(order_request_records):,}")

CCU-Auftragsanalyse
Vollständig beobachtete Aufträge: 132
Dauer = stoppedAt - startedAt; Wartezeit = startedAt - receivedAt
Schrittklassifikation: FINISHED/PASSED = durchgeführt; FAILED/ERROR = fehlgeschlagen; ENQUEUED/WAITING/SKIPPED = nicht durchgeführt

Auftrag: 1abaebd1-fbc0-46d5-a11c-83c63bed9e87
  Typ/Status: RED / FINISHED
  Dauer (s): 45.647 | Gesamt (s): 45.648
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlgeschlagen: keine
  Nicht durchgeführt: keine

Auftrag: f60bbcd2-8259-4bc3-93ca-e2e8548a4250
  Typ/Status: RED / FINISHED
  Dauer (s): 56.249 | Gesamt (s): 56.249
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlgeschlagen: keine
  Nicht durchgeführt: keine

Auftrag: 4c6e1cb9-f9f0-4cae-a3b2-db7c4a805b4a
  Typ/Status: RED / FINISHED
  Dauer (s): 54.294 | Gesamt (s): 54.294
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlgeschlagen: keine
  Nicht durchgeführt: keine

Auftrag: 292b5687-6c01-4488-a325-be2de25475a8
  Typ/S

### Fehlerbedingte Produktionsschritte

Bei Fehleraufträgen werden nachfolgende Schritte im CCU-Snapshot teilweise ebenfalls als `ERROR` markiert, obwohl sie nicht mehr ausgeführt wurden. Deshalb wird die Reihenfolge berücksichtigt: Der erste fehlerhafte Schritt ist der Fehlerpunkt; alle danach liegenden Schritte werden als wegen dieses Fehlers nicht durchgeführt klassifiziert.

In [ ]:
def classify_steps_after_error(order):
    """Trenne ausgeführte, fehlgeschlagene und danach übersprungene Schritte."""
    success_states = {"FINISHED", "COMPLETED", "PASSED", "SUCCESS", "SUCCEEDED"}
    failure_states = {"FAILED", "ERROR", "CANCELLED", "CANCELED", "ABORTED"}
    performed = []
    failed = []
    not_performed_due_to_error = []
    error_found = False

    for index, step in enumerate(order.get("productionSteps") or [], start=1):
        if not isinstance(step, dict):
            continue
        state = str(step.get("state") or "UNKNOWN").upper()
        step_name = step.get("command") or step.get("type") or "UNKNOWN"
        step_info = {
            "position": index,
            "step_id": step.get("id"),
            "step": step_name,
            "type": step.get("type"),
            "command": step.get("command"),
            "module_type": step.get("moduleType"),
            "serial_number": step.get("serialNumber"),
            "state": state,
            "started_at": step.get("startedAt"),
            "stopped_at": step.get("stoppedAt"),
        }

        if error_found:
            not_performed_due_to_error.append(step_info)
        elif state in failure_states:
            # Der erste Fehler beendet den geplanten Ablauf.
            failed.append(step_info)
            error_found = True
        elif state in success_states:
            performed.append(step_info)
        else:
            # Ein unbekannter/warteähnlicher Zustand vor dem Fehler wurde nicht abgeschlossen.
            not_performed_due_to_error.append(step_info)

    return performed, failed, not_performed_due_to_error


# Die bisherige Klassifikation wird mit der Reihenfolge der Produktionsschritte verfeinert.
for row in order_duration_rows:
    order = latest_completed[row["order_id"]]
    performed, failed, not_performed = classify_steps_after_error(order)
    row["performed_steps"] = performed
    row["failed_steps"] = failed
    row["not_performed_steps"] = not_performed
    row["performed_step_names"] = [step["step"] for step in performed]
    row["failed_step_names"] = [step["step"] for step in failed]
    row["not_performed_step_names"] = [step["step"] for step in not_performed]
    row["has_step_error"] = bool(failed)
    row["has_unperformed_steps"] = bool(not_performed)

print("\nProduktionsschritte je Auftrag nach Fehlerpunkt")
for row in order_duration_rows:
    print(f"\nAuftrag {row['order_id']} | Status: {row['state']} | Typ: {row['type']}")
    print("  Durchgeführt:", row["performed_step_names"] or "keine")
    print("  Fehlerpunkt:", row["failed_step_names"] or "kein Fehler")
    print("  Wegen Fehler nicht durchgeführt:", row["not_performed_step_names"] or "keine")


Produktionsschritte je Auftrag nach Fehlerpunkt

Auftrag 1abaebd1-fbc0-46d5-a11c-83c63bed9e87 | Status: FINISHED | Typ: RED
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlerpunkt: kein Fehler
  Wegen Fehler nicht durchgeführt: keine

Auftrag f60bbcd2-8259-4bc3-93ca-e2e8548a4250 | Status: FINISHED | Typ: RED
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlerpunkt: kein Fehler
  Wegen Fehler nicht durchgeführt: keine

Auftrag 4c6e1cb9-f9f0-4cae-a3b2-db7c4a805b4a | Status: FINISHED | Typ: RED
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlerpunkt: kein Fehler
  Wegen Fehler nicht durchgeführt: keine

Auftrag 292b5687-6c01-4488-a325-be2de25475a8 | Status: FINISHED | Typ: WHITE
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 'PICK']
  Fehlerpunkt: kein Fehler
  Wegen Fehler nicht durchgeführt: keine

Auftrag aab5df72-b2ee-4d94-b36e-f249b0aa38e5 | Status: FINISHED | Typ: RED
  Durchgeführt: ['NAVIGATION', 'DROP', 'NAVIGATION', 

## 5. Modulaktionsdauern berechnen

Die Dauer einer Modulaktion wird aus den Action-State-Zeitstempeln berechnet. Für die Auswertung werden Start, tatsächlicher `RUNNING`-Beginn und terminaler Zustand getrennt ausgegeben.

In [ ]:
# Gleiche action_id und denselben Auftrag zu einer Aktion gruppieren.
module_groups = defaultdict(list)
for row in module_action_records:
    key = (row["module"], row["action_id"], row["order_id"], row["command"])
    module_groups[key].append(row)

module_duration_rows = []
for (module, action_id, order_id, command), rows in module_groups.items():
    states = defaultdict(list)
    for row in rows:
        states[row["state"]].append(row["timestamp"])

    start_candidates = [min(states[state]) for state in ("WAITING", "INITIALIZING", "RUNNING") if states.get(state)]
    finish_candidates = [min(states[state]) for state in ("FINISHED", "FAILED") if states.get(state)]
    start_at = min(start_candidates) if start_candidates else None
    end_at = min(finish_candidates) if finish_candidates else None
    running_at = min(states["RUNNING"]) if states.get("RUNNING") else None
    terminal_state = "FAILED" if states.get("FAILED") else "FINISHED" if states.get("FINISHED") else None

    module_duration_rows.append({
        "module": module,
        "action_id": action_id,
        "order_id": order_id,
        "command": command,
        "start_at": start_at,
        "running_at": running_at,
        "end_at": end_at,
        "state": terminal_state,
        "start_to_running_s": seconds_between(start_at, running_at),
        "running_duration_s": seconds_between(running_at, end_at),
        "total_duration_s": seconds_between(start_at, end_at),
        "observed_states": sorted(states),
    })

module_duration_rows.sort(key=lambda row: row["start_at"] or datetime.max.replace(tzinfo=timezone.utc))
usable_module_duration_rows = [
    row for row in module_duration_rows
    if row.get("command") and row.get("action_id") and row.get("total_duration_s") is not None
]

print(f"Gruppierte Modulaktionen: {len(module_duration_rows):,}")
print(f"Verwendbare Aktionen für Dauerstatistik: {len(usable_module_duration_rows):,}")
for row in usable_module_duration_rows[:10]:
    print(compact_row(row, ["module", "command", "state", "start_to_running_s", "running_duration_s", "total_duration_s"]))

Gruppierte Modulaktionen: 884
Verwendbare Aktionen für Dauerstatistik: 650
{'module': 'SVR4H92774', 'command': 'INPUT_RGB', 'state': 'FINISHED', 'start_to_running_s': 0.0, 'running_duration_s': 15.156, 'total_duration_s': 15.156}
{'module': 'SVR4H92774', 'command': 'RGB_NFC', 'state': 'FINISHED', 'start_to_running_s': 0.034, 'running_duration_s': 29.501, 'total_duration_s': 29.534}
{'module': 'SVR4H92774', 'command': 'INPUT_RGB', 'state': 'FINISHED', 'start_to_running_s': 0.438, 'running_duration_s': 15.005, 'total_duration_s': 15.443}
{'module': 'SVR4H92774', 'command': 'DROP', 'state': 'FINISHED', 'start_to_running_s': 0.177, 'running_duration_s': 13.98, 'total_duration_s': 14.157}
{'module': 'SVR4H73901', 'command': 'PICK', 'state': 'FINISHED', 'start_to_running_s': 0.0, 'running_duration_s': 17.302, 'total_duration_s': 17.302}
{'module': 'SVR4H92774', 'command': 'INPUT_RGB', 'state': 'FINISHED', 'start_to_running_s': 0.421, 'running_duration_s': 15.15, 'total_duration_s': 15.571}
{

### Modul-Order-Nachrichten

Diese Nachrichten markieren den Versand eines konkreten Befehls an ein Modul und ergänzen die späteren Zustandswechsel.

In [ ]:
module_order_count = 0
for record in records:
    topic = record["topic"]
    payload = record["payload"]
    timestamp = record_time(record)
    if not (topic.startswith("module/") and topic.endswith("/order") and isinstance(payload, dict) and timestamp):
        continue

    action = payload.get("action") or {}
    if not isinstance(action, dict):
        continue

    module_id = payload.get("serialNumber") or topic.split("/")[-2]
    add_event(
        record=record,
        timestamp=timestamp,
        activity=f"MODULE_ORDER_{action.get('command', 'UNKNOWN')}",
        case_id=payload.get("orderId"),
        resource=module_id,
        order_id=payload.get("orderId"),
        order_update_id=payload.get("orderUpdateId"),
        action_id=action.get("id"),
        event_kind="module_order",
        details={"command": action.get("command"), "metadata": action.get("metadata", {})},
    )
    module_order_count += 1

print(f"Modul-Order-Events: {module_order_count:,}")

Modul-Order-Events: 1,346


## 6. Fehlerereignisse aus Modul- und FTS-States

Fehler werden unabhängig von der Daueranalyse gespeichert. So können lange Prozesse später mit `error_type`, `error_level`, Gerät und `order_id` verknüpft werden.

In [ ]:
# Modulfehler ergänzen; FTS-Fehler wurden bereits bei der FTS-Extraktion gesammelt.
for record in records:
    topic = record["topic"]
    payload = record["payload"]
    timestamp = record_time(record)
    if not (topic.startswith("module/") and topic.endswith("/state") and isinstance(payload, dict) and timestamp):
        continue

    module_id = payload.get("serialNumber") or topic.split("/")[-2]
    for error in payload.get("errors") or []:
        if not isinstance(error, dict):
            continue
        error_time = parse_event_time(error.get("timestamp")) or timestamp
        error_records.append({
            "timestamp": error_time,
            "device": module_id,
            "device_type": "module",
            "error_type": error.get("errorType"),
            "error_level": error.get("errorLevel"),
            "references": error.get("errorReferences", []),
            "order_id": payload.get("orderId"),
            "record": record,
        })
        add_event(
            record=record,
            timestamp=error_time,
            activity=f"ERROR_{error.get('errorType', 'UNKNOWN')}",
            case_id=payload.get("orderId"),
            resource=module_id,
            order_id=payload.get("orderId"),
            event_kind="module_error",
            details={"error_level": error.get("errorLevel"), "references": error.get("errorReferences", [])},
        )

print(f"Fehlerereignisse insgesamt: {len(error_records):,}")
print("Klassifizierte Fehler:", sum(bool(row.get("error_type") or row.get("error_level")) for row in error_records))

Fehlerereignisse insgesamt: 517
Klassifizierte Fehler: 123


## 6. FTS-Fahr- und Übergabeintervalle berechnen

Ein Fahrintervall beginnt bei `driving: False -> True` und endet bei `True -> False`. Ein Load-Handling-Intervall beginnt bei `waitingForLoadHandling: False -> True` und endet beim Zurückwechseln auf `False`.

In [ ]:
fts_by_device = defaultdict(list)
for row in fts_state_records:
    fts_by_device[row["fts"]].append(row)

fts_destinations = {
    (row["fts"], row["order_id"]): {
        "destination_node": row.get("target"),
        "destination_sequence": row.get("target_sequence"),
        "route_node_ids": row.get("node_ids", []),
    }
    for row in fts_order_records
}

fts_transport_rows = []
fts_handling_rows = []

for fts_id, rows in fts_by_device.items():
    rows.sort(key=lambda row: row["timestamp"])
    previous = None
    driving_start = None
    handling_start = None

    for row in rows:
        if previous is not None:
            # Fahrt: Start mit dem ersten beobachteten driving=True.
            if previous["driving"] is False and row["driving"] is True:
                driving_start = row
                destination = fts_destinations.get((fts_id, row["order_id"]), {})
                add_event(
                    record=row["record"],
                    timestamp=row["timestamp"],
                    activity="FTS_DRIVING_STARTED",
                    case_id=row["order_id"],
                    resource=fts_id,
                    order_id=row["order_id"],
                    event_kind="fts_state_transition",
                    destination_node=destination.get("destination_node"),
                    destination_sequence=destination.get("destination_sequence"),
                    current_node=row.get("last_node_id"),
                    current_sequence=row.get("last_node_sequence_id"),
                    location=row.get("last_node_id"),
                    details={
                        "destination_node": destination.get("destination_node"),
                        "destination_sequence": destination.get("destination_sequence"),
                        "route_node_ids": destination.get("route_node_ids", []),
                        "current_node": row.get("last_node_id"),
                        "current_sequence": row.get("last_node_sequence_id"),
                    },
                )

            # Fahrt: Ende beim ersten beobachteten driving=False.
            elif previous["driving"] is True and row["driving"] is False and driving_start is not None:
                destination = fts_destinations.get((fts_id, driving_start["order_id"]), {})
                fts_transport_rows.append({
                    "fts": fts_id,
                    "order_id": driving_start["order_id"],
                    "start_at": driving_start["timestamp"],
                    "end_at": row["timestamp"],
                    "duration_s": seconds_between(driving_start["timestamp"], row["timestamp"]),
                    "start_node": driving_start["last_node_id"],
                    "end_node": row["last_node_id"],
                    "end_node_sequence": row["last_node_sequence_id"],
                    "destination_node": destination.get("destination_node"),
                    "destination_sequence": destination.get("destination_sequence"),
                    "end_reason": "driving_false",
                })
                add_event(
                    record=row["record"],
                    timestamp=row["timestamp"],
                    activity="FTS_DRIVING_STOPPED",
                    case_id=row["order_id"],
                    resource=fts_id,
                    order_id=row["order_id"],
                    event_kind="fts_state_transition",
                    destination_node=destination.get("destination_node"),
                    destination_sequence=destination.get("destination_sequence"),
                    current_node=row.get("last_node_id"),
                    current_sequence=row.get("last_node_sequence_id"),
                    location=row.get("last_node_id"),
                    details={
                        "destination_node": destination.get("destination_node"),
                        "destination_sequence": destination.get("destination_sequence"),
                        "current_node": row.get("last_node_id"),
                        "current_sequence": row.get("last_node_sequence_id"),
                        "arrival_at_destination": row.get("last_node_id") == destination.get("destination_node"),
                    },
                )
                driving_start = None

            # Übergabe: FTS wartet auf PICK/DROP bzw. clearLoadHandler.
            if previous["waiting_for_load_handling"] is False and row["waiting_for_load_handling"] is True:
                handling_start = row
                destination = fts_destinations.get((fts_id, row["order_id"]), {})
                add_event(
                    record=row["record"],
                    timestamp=row["timestamp"],
                    activity="FTS_LOAD_HANDLING_STARTED",
                    case_id=row["order_id"],
                    resource=fts_id,
                    order_id=row["order_id"],
                    event_kind="fts_state_transition",
                    destination_node=destination.get("destination_node"),
                    destination_sequence=destination.get("destination_sequence"),
                    current_node=row.get("last_node_id"),
                    current_sequence=row.get("last_node_sequence_id"),
                    location=row.get("last_node_id"),
                    details={"current_node": row.get("last_node_id"), "current_sequence": row.get("last_node_sequence_id")},
                )
            elif previous["waiting_for_load_handling"] is True and row["waiting_for_load_handling"] is False and handling_start is not None:
                destination = fts_destinations.get((fts_id, handling_start["order_id"]), {})
                fts_handling_rows.append({
                    "fts": fts_id,
                    "order_id": handling_start["order_id"],
                    "start_at": handling_start["timestamp"],
                    "end_at": row["timestamp"],
                    "duration_s": seconds_between(handling_start["timestamp"], row["timestamp"]),
                    "node_id": handling_start["last_node_id"],
                    "destination_node": destination.get("destination_node"),
                    "end_reason": "waitingForLoadHandling_false",
                })
                add_event(
                    record=row["record"],
                    timestamp=row["timestamp"],
                    activity="FTS_LOAD_HANDLING_STOPPED",
                    case_id=row["order_id"],
                    resource=fts_id,
                    order_id=row["order_id"],
                    event_kind="fts_state_transition",
                    destination_node=destination.get("destination_node"),
                    destination_sequence=destination.get("destination_sequence"),
                    current_node=row.get("last_node_id"),
                    current_sequence=row.get("last_node_sequence_id"),
                    location=row.get("last_node_id"),
                    details={"current_node": row.get("last_node_id"), "current_sequence": row.get("last_node_sequence_id")},
                )
                handling_start = None

        previous = row

print(f"Abgeschlossene FTS-Fahrintervalle: {len(fts_transport_rows):,}")
print(f"Abgeschlossene Load-Handling-Intervalle: {len(fts_handling_rows):,}")
print("Beispiel Fahrintervalle:")
for row in fts_transport_rows[:8]:
    print(compact_row(row, ["fts", "order_id", "duration_s", "start_node", "end_node", "destination_node", "destination_sequence"]))
print("Beispiel Übergabeintervalle:")
for row in fts_handling_rows[:8]:
    print(compact_row(row, ["fts", "order_id", "duration_s", "node_id", "destination_node"]))


Abgeschlossene FTS-Fahrintervalle: 455
Abgeschlossene Load-Handling-Intervalle: 425
Beispiel Fahrintervalle:
{'fts': 'ylK4', 'order_id': '1abaebd1-fbc0-46d5-a11c-83c63bed9e87', 'duration_s': 1.296, 'start_node': 'SVR4H92774', 'end_node': 'SVR4H92774', 'destination_node': 'SVR4H73901', 'destination_sequence': 4}
{'fts': 'ylK4', 'order_id': '1abaebd1-fbc0-46d5-a11c-83c63bed9e87', 'duration_s': 10.849, 'start_node': 'SVR4H92774', 'end_node': 'SVR4H73901', 'destination_node': 'SVR4H73901', 'destination_sequence': 4}
{'fts': 'ylK4', 'order_id': 'f60bbcd2-8259-4bc3-93ca-e2e8548a4250', 'duration_s': 12.192, 'start_node': 'SVR4H73901', 'end_node': 'SVR4H92774', 'destination_node': 'SVR4H73901', 'destination_sequence': 4}
{'fts': 'ylK4', 'order_id': 'f60bbcd2-8259-4bc3-93ca-e2e8548a4250', 'duration_s': 11.725, 'start_node': 'SVR4H92774', 'end_node': 'SVR4H73901', 'destination_node': 'SVR4H73901', 'destination_sequence': 4}
{'fts': 'ylK4', 'order_id': '4c6e1cb9-f9f0-4cae-a3b2-db7c4a805b4a', 'dur

In [ ]:
# FTS-Aktivitäten um Ortsinformationen ergänzen.
# Die ursprünglichen Aktivitätsnamen bleiben als Präfix erhalten; Knoten-IDs
# werden unverändert aus den MQTT-Daten übernommen.
def fts_location_label(value):
    text = str(value or "UNKNOWN_LOCATION").strip()
    return text.replace(" ", "_").replace("/", "_")


def enrich_fts_activity(event):
    if not str(event.get("event_kind") or "").startswith("fts_"):
        return

    activity = str(event.get("activity") or "FTS_EVENT")
    location = event.get("location") or event.get("current_node")
    destination = event.get("destination_node")

    if event.get("event_kind") == "fts_order" and destination and "_TO_" not in activity:
        event["activity"] = f"{activity}_TO_{fts_location_label(destination)}"
    elif event.get("event_kind") == "fts_state_transition":
        location_label = fts_location_label(location)
        destination_label = fts_location_label(destination) if destination else None
        if "DRIVING_STARTED" in activity and destination_label and "_FROM_" not in activity:
            event["activity"] = f"{activity}_FROM_{location_label}_TO_{destination_label}"
        elif "DRIVING_STOPPED" in activity and destination_label and "_AT_" not in activity:
            event["activity"] = f"{activity}_AT_{location_label}_FROM_{destination_label}"
        elif "LOAD_HANDLING" in activity and "_AT_" not in activity:
            event["activity"] = f"{activity}_AT_{location_label}"
    elif event.get("event_kind") == "fts_error" and location and "_AT_" not in activity:
        event["activity"] = f"{activity}_AT_{fts_location_label(location)}"


for event in event_records:
    enrich_fts_activity(event)

# Dauer und Haltepunkt an die jeweils abschliessenden FTS-Ereignisse schreiben.
for event in event_records:
    if event.get("event_kind") != "fts_state_transition":
        continue
    details = event.setdefault("details", {})
    if "DRIVING_STOPPED" in str(event.get("activity") or ""):
        matches = [
            row for row in fts_transport_rows
            if row.get("fts") == event.get("resource")
            and row.get("order_id") == event.get("order_id")
            and row.get("end_at") == event.get("timestamp")
        ]
        if matches:
            route = matches[0]
            details.update({
                "duration_s": route.get("duration_s"),
                "route_start_node": route.get("start_node"),
                "route_end_node": route.get("end_node"),
                "destination_node": route.get("destination_node"),
            })
    elif "LOAD_HANDLING_STOPPED" in str(event.get("activity") or ""):
        matches = [
            row for row in fts_handling_rows
            if row.get("fts") == event.get("resource")
            and row.get("order_id") == event.get("order_id")
            and row.get("end_at") == event.get("timestamp")
        ]
        if matches:
            handling = matches[0]
            details.update({
                "duration_s": handling.get("duration_s"),
                "pitstop_node": handling.get("node_id"),
                "destination_node": handling.get("destination_node"),
            })

print("Ortsbezogene FTS-Aktivitäten:")
for event in [event for event in event_records if str(event.get("event_kind") or "").startswith("fts_")][:20]:
    print(event.get("activity"), event.get("details", {}).get("duration_s"))

Ortsbezogene FTS-Aktivitäten:
FTS_ORDER_SENT_TO_SVR4H92774 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H73901 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H92774 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H73901 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H92774 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H73901 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H92774 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H73901 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H92774 None
FTS_clearLoadHandler None
FTS_ORDER_SENT_TO_SVR4H73901 None
FTS_clearLoadHandler None


In [ ]:
def percentile(values, fraction):
    """Berechne ein Quantil ohne externe Bibliotheken."""
    values = sorted(value for value in values if value is not None)
    if not values:
        return None
    position = (len(values) - 1) * fraction
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    weight = position - lower
    return round(values[lower] + (values[upper] - values[lower]) * weight, 3)


def print_outliers(title, rows, value_key, threshold, fields):
    """Drucke die längsten Beobachtungen oberhalb der p95-Schwelle."""
    outliers = [row for row in rows if row.get(value_key) is not None and row[value_key] > threshold]
    print(f"\n{title}")
    print(f"p95-Schwelle: {threshold}s | Ausreißer: {len(outliers):,}")
    for row in sorted(outliers, key=lambda item: item[value_key], reverse=True)[:15]:
        print(compact_row(row, fields + [value_key]))
    return outliers


# Konfiguration als fachliche Referenz für Produktionsdauern laden.
configured_durations = {}
for record in records:
    if record["topic"] == "ccu/state/config" and isinstance(record["payload"], dict):
        values = record["payload"].get("productionDurations") or {}
        if isinstance(values, dict):
            configured_durations.update({key: value for key, value in values.items() if isinstance(value, (int, float))})

for row in order_duration_rows:
    expected = configured_durations.get(row.get("type"))
    row["configured_production_duration_s"] = expected
    row["deviation_from_configured_s"] = (
        round(row["processing_duration_s"] - expected, 3)
        if expected is not None and row["processing_duration_s"] is not None else None
    )
    row["ratio_to_configured"] = (
        round(row["processing_duration_s"] / expected, 3)
        if expected and row["processing_duration_s"] is not None else None
    )

print("Konfigurierte Produktionsdauern:", configured_durations)
print("\nGrößte Abweichungen von der Konfiguration:")
for row in sorted(order_duration_rows, key=lambda item: item.get("ratio_to_configured") or -1, reverse=True)[:10]:
    print(compact_row(row, ["order_id", "type", "state", "processing_duration_s", "configured_production_duration_s", "deviation_from_configured_s", "ratio_to_configured"]))

# p95 ist hier ein datengetriebener Warnwert, keine harte technische Spezifikation.
order_p95 = percentile([row["total_duration_s"] for row in order_duration_rows], 0.95)
module_p95 = percentile([row["total_duration_s"] for row in usable_module_duration_rows], 0.95)
transport_p95 = percentile([row["duration_s"] for row in fts_transport_rows], 0.95)
handling_p95 = percentile([row["duration_s"] for row in fts_handling_rows], 0.95)

order_outliers = print_outliers("Lange Gesamtaufträge", order_duration_rows, "total_duration_s", order_p95, ["order_id", "type", "state"])
module_outliers = print_outliers("Lange Modulaktionen", usable_module_duration_rows, "total_duration_s", module_p95, ["module", "command", "order_id", "state"])
transport_outliers = print_outliers("Lange FTS-Fahrintervalle", fts_transport_rows, "duration_s", transport_p95, ["fts", "order_id", "start_node", "end_node"])
handling_outliers = print_outliers("Langes FTS-Load-Handling", fts_handling_rows, "duration_s", handling_p95, ["fts", "order_id", "node_id"])

classified_errors = [row for row in error_records if row.get("error_type") or row.get("error_level")]
unclassified_errors = [row for row in error_records if not row.get("error_type") and not row.get("error_level")]
print("\nFehlerqualität")
print(f"Klassifizierte Fehler: {len(classified_errors):,}")
print(f"Unklassifizierte Fehlerobjekte: {len(unclassified_errors):,}")
for row in classified_errors[:15]:
    print(compact_row(row, ["timestamp", "device", "device_type", "error_type", "error_level", "order_id"]))

# Datenqualität: fehlende Endpunkte müssen vor einer späteren Event-Log-Erzeugung sichtbar bleiben.
print("\nDatenqualität")
print("Aufträge ohne Start/Ende:", sum(row["started_at"] is None or row["stopped_at"] is None for row in order_duration_rows))
print("Modulaktionen ohne terminalen Zustand:", sum(row["end_at"] is None for row in usable_module_duration_rows))
print("FTS-Fahrintervalle ohne Ende:", sum(row["end_at"] is None for row in fts_transport_rows))
print("FTS-Handlingintervalle ohne Ende:", sum(row["end_at"] is None for row in fts_handling_rows))

# Ressourcen aus dem Topic ergänzen, falls ein Modul-Order-Payload keine serialNumber enthält.
for event in event_records:
    if event["event_kind"] == "module_order" and event["resource"] is None:
        parts = event["source_topic"].split("/")
        event["resource"] = parts[-2] if len(parts) >= 2 else None

event_records.sort(key=lambda row: row["timestamp"] or datetime.max.replace(tzinfo=timezone.utc))
print("\nEvent-Log-kompatible Vorschau")
print("Spalten: case_id, activity, timestamp, resource, lifecycle, order_id, action_id, event_kind")
for row in event_records[:12]:
    print({key: iso(row[key]) if key == "timestamp" else row[key] for key in ("case_id", "activity", "timestamp", "resource", "lifecycle", "order_id", "action_id", "event_kind")})

print("\nEvent-Mengen nach Typ")
for event_kind, count in Counter(event["event_kind"] for event in event_records).most_common():
    print(f"{count:>6,}  {event_kind}")

Konfigurierte Produktionsdauern: {'WHITE': 580, 'BLUE': 580, 'RED': 580}

Größte Abweichungen von der Konfiguration:
{'order_id': 'bffbc0c7-3e4d-4418-aca8-1d1627481111', 'type': 'BLUE', 'state': 'FINISHED', 'processing_duration_s': 780.211, 'configured_production_duration_s': 580, 'deviation_from_configured_s': 200.211, 'ratio_to_configured': 1.345}
{'order_id': '74fe95a4-6dd7-40db-8e02-8617999e3fc1', 'type': 'WHITE', 'state': 'FINISHED', 'processing_duration_s': 753.321, 'configured_production_duration_s': 580, 'deviation_from_configured_s': 173.321, 'ratio_to_configured': 1.299}
{'order_id': 'e0b230ba-0ba0-45dc-8931-a47c7b98849a', 'type': 'BLUE', 'state': 'ERROR', 'processing_duration_s': 696.907, 'configured_production_duration_s': 580, 'deviation_from_configured_s': 116.907, 'ratio_to_configured': 1.202}
{'order_id': '9fff8396-6547-48f1-9d1a-ad7e2618650f', 'type': 'BLUE', 'state': 'ERROR', 'processing_duration_s': 662.901, 'configured_production_duration_s': 580, 'deviation_from_co

## 7. Event Log erzeugen

Der Event Log beschreibt beobachtete Abläufe als Start-/End-Paare. Intervallereignisse erhalten deshalb die Namen `EVENTNAME_start` und `EVENTNAME_end`; punktuelle Ereignisse werden mit gleichem Zeitstempel als Start und Ende ausgegeben. Jeder Datensatz behält die wichtigste Fall-, Ressourcen- und Quelleninformation.

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
from xml.sax.saxutils import escape
from collections import defaultdict


def event_timestamp(value):
    """Format a datetime value according to the XES timestamp format."""
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def normalize_case_label(value):
    """Ein Case entspricht in diesem Projekt einer Original-JSONL-Datei (z. B. Ablauf1 100626)."""
    if value is None:
        return "UNKNOWN_CASE"
    text = str(value).strip()
    if not text:
        return "UNKNOWN_CASE"
    if text.lower().endswith((".jsonl", ".json")):
        return Path(text).stem
    if "\\" in text or "/" in text:
        stem = Path(text).stem
        return stem or text
    return text


def case_key_for_event(event):
    source_file = event.get("source_file") or event.get("file")
    if source_file:
        return normalize_case_label(source_file)

    case_value = (
        event.get("case_id")
        or event.get("order_id")
        or event.get("resource")
        or "UNKNOWN_CASE"
    )
    return normalize_case_label(case_value)


output_dir = Path("Zeitdatenanlyse")
output_dir.mkdir(exist_ok=True)
xes_path = output_dir / "event_log.xes"
for stale_path in (output_dir / "event_log.csv", output_dir / "event_log.json"):
    if stale_path.exists():
        stale_path.unlink()

# Immer den aktuell aufgebauten Event-Datensatz exportieren, nicht einen alten Kernel-Export.
source_events = event_records
filtered_event_log = [
    event for event in source_events
    if "factsheet" not in str(event.get("activity") or "").lower()
]

normalized_event_log = []
for event in filtered_event_log:
    normalized = dict(event)
    normalized["case_id"] = case_key_for_event(event)
    normalized_event_log.append(normalized)

normalized_event_log.sort(key=lambda event: (
    event["timestamp"] is None,
    event["timestamp"] or datetime.max.replace(tzinfo=timezone.utc),
    event.get("lifecycle") != "start",
    event.get("case_id") or "",
    event.get("activity") or "",
))

case_groups = defaultdict(list)
for event in normalized_event_log:
    case_name = str(event.get("case_id") or "UNKNOWN_CASE")
    case_groups[case_name].append(event)

with xes_path.open("w", encoding="utf-8") as output_file:
    output_file.write('<?xml version="1.0" encoding="UTF-8"?>\n')
    output_file.write('<log xes.version="1.0" xes.features="nested-attributes" openxes.version="1.0RC7" xmlns="http://www.xes-standard.org/">\n')
    output_file.write('  <extension name="Concept" prefix="concept" uri="http://www.xes-standard.org/concept.xesext"/>\n')
    output_file.write('  <extension name="Time" prefix="time" uri="http://www.xes-standard.org/time.xesext"/>\n')
    output_file.write('  <extension name="Organizational" prefix="org" uri="http://www.xes-standard.org/org.xesext"/>\n')
    output_file.write('  <global scope="trace"><string key="concept:name" value=""/></global>\n')
    output_file.write('  <global scope="event"><string key="concept:name" value=""/><date key="time:timestamp" value=""/></global>\n')
    output_file.write('  <classifier name="Event Name" keys="concept:name"/>\n')
    output_file.write('  <string key="description" value="FT MQTT production event log with FTS destinations and locations"/>\n')

    for case_name in sorted(case_groups):
        events = sorted(case_groups[case_name], key=lambda event: (
            event["timestamp"] is None,
            event["timestamp"] or datetime.max.replace(tzinfo=timezone.utc),
            event["lifecycle"] != "start",
            event.get("activity") or "",
        ))

        output_file.write("  <trace>\n")
        output_file.write(f'    <string key="concept:name" value="{escape(case_name, {chr(34): "&quot;"})}"/>\n')

        for event in events:
            details = event.get("details") or {}
            details_text = json.dumps(details, ensure_ascii=False, default=str, separators=(",", ":")) if details else ""

            output_file.write("    <event>\n")
            output_file.write(f'      <string key="concept:name" value="{escape(str(event.get("activity") or "EVENT"), {chr(34): "&quot;"})}"/>\n')
            if event.get("timestamp") is not None:
                output_file.write(f'      <date key="time:timestamp" value="{event_timestamp(event["timestamp"])}"/>\n')
            if event.get("resource") is not None:
                output_file.write(f'      <string key="org:resource" value="{escape(str(event["resource"]), {chr(34): "&quot;"})}"/>\n')
            if event.get("lifecycle") is not None:
                output_file.write(f'      <string key="lifecycle" value="{escape(str(event["lifecycle"]), {chr(34): "&quot;"})}"/>\n')
            for key in ("order_id", "action_id", "event_kind", "source_file", "source_line",
                        "destination_node", "destination_sequence", "current_node",
                        "current_sequence", "location"):
                value = event.get(key)
                if value is not None and value != "":
                    output_file.write(f'      <string key="{key}" value="{escape(str(value), {chr(34): "&quot;"})}"/>\n')
            if details_text:
                output_file.write(f'      <string key="details" value="{escape(details_text, {chr(34): "&quot;"})}"/>\n')
            output_file.write("    </event>\n")

        output_file.write("  </trace>\n")

    output_file.write("</log>\n")

print(f"XES Log erzeugt: {xes_path.resolve()}")
print(f"Traces: {len(case_groups):,}")
print(f"Events: {len(normalized_event_log):,}")
print(f"Factsheet-Events entfernt: {len(filtered_event_log) - len(normalized_event_log):,}")
print("FTS-Events enthalten Zielknoten, Zielsequenz, aktuellen Knoten und aktuellen Standort als XES-Attribute.")
print("Case-Definition: ein Case = ein Ablauf / eine Original-JSONL-Datei (z.B. Ablauf1 100626, Ablauf2 100626).")
print("CSV/JSON-Export deaktiviert; nur XES wird gespeichert.")
print("\nVorschau:")
for event in normalized_event_log[:12]:
    print({field: event.get(field) for field in ("case_id", "activity", "timestamp", "resource", "destination_node", "current_node", "location", "event_kind")})


XES Log erzeugt: C:\Users\olive\Documents\Uni\Forschungsprojekt\Zeitdatenanlyse\event_log.xes
Traces: 13
Events: 26,084
Factsheet-Events entfernt: 0
FTS-Events enthalten Zielknoten, Zielsequenz, aktuellen Knoten und aktuellen Standort als XES-Attribute.
Case-Definition: ein Case = ein Ablauf / eine Original-JSONL-Datei (z.B. Ablauf1 100626, Ablauf2 100626).
CSV/JSON-Export deaktiviert; nur XES wird gespeichert.

Vorschau:
{'case_id': 'test lagerplatzbelegung 230326', 'activity': 'MODULE_INPUT_RGB_RUNNING', 'timestamp': datetime.datetime(2026, 3, 23, 9, 28, 36, 90563, tzinfo=datetime.timezone.utc), 'resource': 'SVR4H92774', 'destination_node': None, 'current_node': None, 'location': None, 'event_kind': 'module_action_state'}
{'case_id': 'test lagerplatzbelegung 230326', 'activity': 'MODULE_INPUT_RGB_FINISHED', 'timestamp': datetime.datetime(2026, 3, 23, 9, 28, 51, 246937, tzinfo=datetime.timezone.utc), 'resource': 'SVR4H92774', 'destination_node': None, 'current_node': None, 'location':

In [ ]:
import xml.etree.ElementTree as ET

xes_tree = ET.parse(xes_path)
xes_root = xes_tree.getroot()
xes_events = xes_root.findall(".//{http://www.xes-standard.org/}event")
xes_destination_events = [
    event for event in xes_events
    if any(attribute.get("key") == "destination_node" for attribute in event)
]
print(f"XES XML valide: {xes_path.name}")
print(f"XES-Events geprüft: {len(xes_events):,}")
print(f"Events mit destination_node: {len(xes_destination_events):,}")
print("Factsheet im XES:", any("factsheet" in (attribute.get("value") or "").lower() for attribute in xes_root.iter()))


XES XML valide: event_log.xes
XES-Events geprüft: 26,084
Events mit destination_node: 2,403
Factsheet im XES: False


In [ ]:
# DFG (Directly-Follows Graph) nur für die Hauptprozesse der Ablaufdateien erzeugen.
# Die Aktivitäten stammen aus normalized_event_log und enthalten damit auch die
# ortsbezogenen FTS-Namen aus der vorherigen Analysezelle.
import re
from collections import defaultdict

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import networkx as nx
except ImportError:
    raise RuntimeError("Bitte matplotlib und networkx installieren: pip install matplotlib networkx")


def normalize_activity(activity):
    text = str(activity or "UNKNOWN")
    text = re.sub(r"(_start|_end)$", "", text)
    return text.replace("MODULE_factsheetRequest_", "").replace("MODULE_", "").replace("FTS_", "")


def normalize_case_label(value):
    if value is None:
        return "UNKNOWN_CASE"
    text = str(value).strip()
    if not text:
        return "UNKNOWN_CASE"
    if text.lower().endswith((".jsonl", ".json")):
        return Path(text).stem
    if "\\" in text or "/" in text:
        stem = Path(text).stem
        return stem or text
    return text


def is_noise_activity(activity):
    text = str(activity or "").upper()
    return "FACTSHEET" in text or "ERROR_" in text or text.startswith("FTS_INSTANT")


def build_case_df_graph(case_events):
    ordered = sorted(
        case_events,
        key=lambda event: (
            event["timestamp"] is None,
            event["timestamp"] or datetime.max.replace(tzinfo=timezone.utc),
            event.get("lifecycle") != "start",
            event.get("activity") or "",
        ),
    )

    transitions = defaultdict(list)
    for previous, current in zip(ordered, ordered[1:]):
        prev_ts = previous.get("timestamp")
        curr_ts = current.get("timestamp")
        if prev_ts is None or curr_ts is None:
            continue

        prev_activity = normalize_activity(previous.get("activity"))
        curr_activity = normalize_activity(current.get("activity"))
        if is_noise_activity(previous.get("activity")) or is_noise_activity(current.get("activity")):
            continue
        if not prev_activity or not curr_activity or prev_activity == curr_activity:
            continue

        elapsed = (curr_ts - prev_ts).total_seconds()
        if elapsed < 0:
            continue
        transitions[(prev_activity, curr_activity)].append(elapsed)

    if not transitions:
        return None

    graph = nx.DiGraph()
    for (source, target), durations in transitions.items():
        mean_seconds = sum(durations) / len(durations)
        graph.add_edge(source, target, mean_time_s=mean_seconds, count=len(durations), label=f"{mean_seconds:.1f}s")

    if graph.number_of_edges() > 25:
        edge_scores = sorted(
            ((data["mean_time_s"], u, v) for u, v, data in graph.edges(data=True)),
            reverse=True,
        )
        keep = {(u, v) for _, u, v in edge_scores[:25]}
        for u, v in list(graph.edges()):
            if (u, v) not in keep:
                graph.remove_edge(u, v)

    return graph


def draw_case_df_graph(case_id, case_events, output_dir):
    graph = build_case_df_graph(case_events)
    if graph is None or graph.number_of_edges() == 0:
        print(f"Case {case_id}: Keine gültigen Directly-Follows-Transitions gefunden.")
        return

    pos = nx.spring_layout(graph, k=1.8, seed=42)
    all_times = [data["mean_time_s"] for _, _, data in graph.edges(data=True)]
    min_time = min(all_times)
    max_time = max(all_times)
    widths = (
        [2.8 for _ in graph.edges()]
        if max_time == min_time
        else [1.8 + 8.0 * ((data["mean_time_s"] - min_time) / (max_time - min_time)) for _, _, data in graph.edges(data=True)]
    )
    edge_labels = {(u, v): f"{data['mean_time_s']:.0f}s" for u, v, data in graph.edges(data=True)}

    fig, ax = plt.subplots(figsize=(18, 10))
    nx.draw_networkx_nodes(graph, pos, node_color="#DBEAFE", node_size=1800, edgecolors="#1F2937", linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(graph, pos, font_size=10, font_weight="bold", ax=ax)
    nx.draw_networkx_edges(graph, pos, width=widths, edge_color="#2563EB", arrows=True, arrowstyle="-|>", arrowsize=18, connectionstyle="arc3,rad=0.08", ax=ax)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=9, rotate=False, bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#9CA3AF", "alpha": 0.9}, ax=ax)
    ax.set_title(f"DFG - Hauptprozess: {case_id}\nÜbergänge mit durchschnittlicher Dauer", fontsize=15)
    ax.axis("off")

    safe_case = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(case_id))[:80] or "unknown_case"
    png_path = output_dir / f"{safe_case}_dfg.png"
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Hauptprozess {case_id}: {graph.number_of_nodes()} Nodes, {graph.number_of_edges()} Kanten -> {png_path.name}")


# Ausschliesslich die Hauptprozesse aus den Original-JSONL-Dateien verwenden.
main_process_events = [
    event for event in normalized_event_log
    if re.match(r"^Ablauf.*\.jsonl$", Path(str(event.get("source_file") or "")).name, re.IGNORECASE)
    and "factsheet" not in str(event.get("activity") or "").lower()
]

dfg_dir = output_dir / "dfg"
dfg_dir.mkdir(exist_ok=True)

main_process_groups = defaultdict(list)
for event in main_process_events:
    case_id = normalize_case_label(event.get("source_file"))
    if case_id != "UNKNOWN_CASE":
        main_process_groups[case_id].append(event)

for case_id, case_events in sorted(main_process_groups.items(), key=lambda item: item[0]):
    draw_case_df_graph(case_id, case_events, dfg_dir)

print(f"\nDFG-Export abgeschlossen: {dfg_dir.resolve()}")
print(f"Hauptprozess-Graphen erzeugt: {len(main_process_groups):,}")
print("Verwendete Ablaufdateien:", ", ".join(sorted(main_process_groups)))

Hauptprozess Ablauf1 100626: 84 Nodes, 25 Kanten -> Ablauf1_100626_dfg.png
Hauptprozess Ablauf2 100626: 70 Nodes, 25 Kanten -> Ablauf2_100626_dfg.png
Hauptprozess Ablauf2.1 08072026: 74 Nodes, 25 Kanten -> Ablauf2.1_08072026_dfg.png
Hauptprozess Ablauf3 100626: 65 Nodes, 25 Kanten -> Ablauf3_100626_dfg.png
Hauptprozess Ablauf4 100626: 75 Nodes, 25 Kanten -> Ablauf4_100626_dfg.png

DFG-Export abgeschlossen: C:\Users\olive\Documents\Uni\Forschungsprojekt\Zeitdatenanlyse\dfg
Hauptprozess-Graphen erzeugt: 5
Verwendete Ablaufdateien: Ablauf1 100626, Ablauf2 100626, Ablauf2.1 08072026, Ablauf3 100626, Ablauf4 100626
